# IndexCalc — Phase 3 테스트

Contraction 분석, Trace, Einstein convention 검증, 표현식 요약.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

from indexcalc import (
    IndexSpace, Tensor, IndexRegistry, parse,
    validate_einstein, collect_tensors, collect_all_indices,
    trace, summarize,
)

In [2]:
# Setup
spacetime = IndexSpace("spacetime", dim=4, indices="μνλρσ", metric="g")
lorentz   = IndexSpace("lorentz",   dim=4, indices="abcde", metric="η")

reg = IndexRegistry()
reg.register(spacetime)
reg.register(lorentz)

## 1. validate_einstein — 전역 인덱스 분석

In [3]:
# 정상: T^μ_ν S^ν_λ
expr = parse("T^{μ}_{ν} S^{ν}_{λ}", reg)
info = validate_einstein(expr)

print(f"Expression: {expr}")
print(f"Valid: {info['valid']}")
print(f"Free: {info['free']}")
print(f"Contracted: {info['contracted']}")

Expression: (T^μ_ν * S^ν_λ)  [contracted: ν]
Valid: True
Free: [^μ, _λ]
Contracted: [(^ν, _ν)]


In [4]:
# 3개 텐서 연쇄 contraction: T^μ_ν g^{νλ} S_{λρ}
expr = parse("T^{μ}_{ν} g^{νλ} S_{λρ}", reg)
info = validate_einstein(expr)

print(f"Expression: {expr}")
print(f"Free: {info['free']}")
print(f"Contracted: {[(a.name, a.space.name) for a, b in info['contracted']]}")

Expression: ((T^μ_ν * g^ν^λ)  [contracted: ν] * S_λ_ρ)  [contracted: λ]
Free: [^μ, _ρ]
Contracted: [('ν', 'spacetime'), ('λ', 'spacetime')]


In [5]:
# 위반: 같은 위치 반복 (둘 다 upper)
bad = Tensor("A", [spacetime.upper("μ"), spacetime.lower("ν")]) * \
      Tensor("B", [spacetime.upper("μ"), spacetime.lower("λ")])
info = validate_einstein(bad)

print(f"Expression: {bad}")
print(f"Valid: {info['valid']}")
for err in info['errors']:
    print(f"  ⚠ {err}")

Expression: (A^μ_ν * B^μ_λ)
Valid: False
  ⚠ Index 'μ' (spacetime) appears twice with same position 'upper'. Contraction requires one upper and one lower.


## 2. collect_tensors & collect_all_indices

In [6]:
expr = parse("T^{μ}_{ν} g^{νλ} S_{λρ}", reg)

print(f"Expression: {expr}")
print(f"Tensors: {collect_tensors(expr)}")
print(f"All indices: {collect_all_indices(expr)}")

Expression: ((T^μ_ν * g^ν^λ)  [contracted: ν] * S_λ_ρ)  [contracted: λ]
Tensors: [T^μ_ν, g^ν^λ, S_λ_ρ]
All indices: [^μ, _ν, ^ν, ^λ, _λ, _ρ]


## 3. Trace — 같은 텐서 내 contraction

In [7]:
# T^μ_μ → scalar
T = Tensor("T", [spacetime.upper("μ"), spacetime.lower("μ")])
tr = trace(T, "μ")

print(f"T = {T}")
print(f"Tr(T) = {tr}")
print(f"Rank: {tr.rank}  (scalar)")

T = T^μ_μ
Tr(T) = Tr(T^μ_μ)
Rank: (0, 0)  (scalar)


In [8]:
# Riemann-like: R^μ_ν^ν_λ → trace over ν
R = Tensor("R", [
    spacetime.upper("μ"),
    spacetime.lower("ν"),
    spacetime.upper("ν"),
    spacetime.lower("λ"),
])
tr_R = trace(R, "ν")

print(f"R = {R}")
print(f"Tr_ν(R) = {tr_R}")
print(f"Free: {tr_R.free_indices}")
print(f"Rank: {tr_R.rank}")

R = R^μ_ν^ν_λ
Tr_ν(R) = Tr_ν(R^μ_ν^ν_λ)
Free: [^μ, _λ]
Rank: (1, 1)


In [9]:
# Trace 에러: 같은 위치 2개
try:
    bad_tensor = Tensor("X", [spacetime.upper("μ"), spacetime.upper("μ")])
    trace(bad_tensor, "μ")
except ValueError as e:
    print(f"예상된 에러: {e}")

예상된 에러: Trace requires one upper and one lower 'μ', but both are upper in X^μ^μ


## 4. summarize — 표현식 한눈에 보기

In [10]:
# Vielbein 곱
print(summarize(parse("e^{a}_{μ} e^{b}^{μ}", reg)))

Expression: (e^a_μ * e^b^μ)  [contracted: μ]
Tensors:    [e^a_μ, e^b^μ]
Free:       ^a, ^b
Contracted: μ (spacetime)
Rank:       (2, 0)


In [11]:
# 연쇄 contraction
print(summarize(parse("T^{μ}_{ν} g^{νλ} S_{λρ}", reg)))

Expression: ((T^μ_ν * g^ν^λ)  [contracted: ν] * S_λ_ρ)  [contracted: λ]
Tensors:    [T^μ_ν, g^ν^λ, S_λ_ρ]
Free:       ^μ, _ρ
Contracted: ν (spacetime), λ (spacetime)
Rank:       (1, 1)


In [12]:
# Metric lowering: g_{μν} V^ν → V_μ
print(summarize(parse("g_{μν} V^{ν}", reg)))

Expression: (g_μ_ν * V^ν)  [contracted: ν]
Tensors:    [g_μ_ν, V^ν]
Free:       _μ
Contracted: ν (spacetime)
Rank:       (0, 1)
